# Module 3 · Integration 3 — ETL Pipeline (Second Build)
**Public Library System (SI Demo)**

This notebook walks through a complete **Extract → Transform → Validate → Load** pipeline for a public library dataset.
It demonstrates the same design pattern you will apply in the Amman Digital Market assignment.

| Step | Function | What it does |
|------|----------|--------------|
| Extract | `extract(engine)` | Pull 3 tables from PostgreSQL into DataFrames |
| Transform | `transform(data_dict)` | Join, compute `loan_duration_days` and `overdue_rate`, filter bad data, aggregate |
| Validate | `validate(df)` | Assert quality rules — raises `ValueError` on failure, warns on soft issues |
| Load | `load(df, engine, csv_path)` | Write to Postgres + CSV, then verify row count |

**Improvements over first build:**
- Added `overdue_rate` metric in transform
- Per-step timing in `run_pipeline`
- Richer data exploration (dtypes + null counts)
- Soft NULL-city warning in validate
- Post-load row count verification
- Pipeline summary at completion

---

### Before you start
1. Make sure PostgreSQL is running locally.
2. Run `seed_db.py` once to populate the source tables.
3. Work through the cells **in order** — each section builds on the previous one.

## 0 · Environment Setup

In [1]:
%pip install -q pandas sqlalchemy psycopg2-binary python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import time
import warnings
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

# ── Load credentials from .env ─────────────────────────────────────────────
load_dotenv()  # reads .env in the current directory

DB_USER     = os.getenv("DB_USER", "postgres")
DB_PASSWORD = os.getenv("DB_PASSWORD", "")
DB_HOST     = os.getenv("DB_HOST", "localhost")
DB_PORT     = int(os.getenv("DB_PORT", "5433"))
DB_NAME     = os.getenv("DB_NAME", "library_demo")

DB_URL = URL.create(
    drivername="postgresql+psycopg2",
    username=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
    database=DB_NAME,
)
engine = create_engine(DB_URL)

# Quick connectivity test
with engine.connect() as conn:
    result = conn.execute(text("SELECT current_database()"))
    print("✅ Connected to:", result.scalar())

/usr/lib/python3/dist-packages/pytz/__init__.py:31: SyntaxWarning: invalid escape sequence '\s'
  match = re.match("^#\s*version\s*([0-9a-z]*)\s*$", line)


✅ Connected to: library_demo


## 1 · Explore the Source Data

Before writing any pipeline code, always understand the raw data.
Run the cells below to inspect each table — including dtypes and null counts.

In [3]:
# Preview all three tables with shape, dtypes, and null counts
with engine.connect() as conn:
    for table in ["members", "books", "loans"]:
        df = pd.read_sql(text(f"SELECT * FROM {table}"), conn)
        print(f"\n── {table}  ({df.shape[0]:,} rows × {df.shape[1]} cols) ──")
        display(df.head(5))
        null_counts = df.isnull().sum()
        if null_counts.any():
            print("  Nulls per column:")
            print(null_counts[null_counts > 0].to_string())
        print(f"  dtypes: {df.dtypes.to_dict()}")


── members  (100 rows × 4 cols) ──


,member_id,name,city,join_date
0,1,Member_001,Zarqa,2018-02-21
1,2,Member_002,Salt,2019-05-17
2,3,Member_003,Aqaba,2018-10-13
3,4,Member_004,Zarqa,2021-10-17
4,5,Member_005,Zarqa,2021-04-24


  Nulls per column:
city    30
  dtypes: {'member_id': dtype('int64'), 'name': <StringDtype(storage='python', na_value=nan)>, 'city': <StringDtype(storage='python', na_value=nan)>, 'join_date': dtype('O')}

── books  (30 rows × 4 cols) ──


,book_id,title,genre,year_published
0,1,The Alchemist,Fiction,1988
1,2,Sapiens,History,2011
2,3,A Brief History of Time,Science,1988
3,4,The Great Gatsby,Fiction,1925
4,5,"Thinking, Fast and Slow",Non-Fiction,2011


  dtypes: {'book_id': dtype('int64'), 'title': <StringDtype(storage='python', na_value=nan)>, 'genre': <StringDtype(storage='python', na_value=nan)>, 'year_published': dtype('int64')}

── loans  (455 rows × 6 cols) ──


,loan_id,member_id,book_id,loan_date,return_date,days_overdue
0,1,1,4,2022-08-23,2022-09-03,0
1,2,1,7,2022-12-05,2022-12-16,0
2,3,1,11,2022-10-13,2022-11-20,24
3,4,1,18,2022-05-16,2022-06-28,29
4,5,1,7,2022-09-06,2022-10-13,23


  dtypes: {'loan_id': dtype('int64'), 'member_id': dtype('int64'), 'book_id': dtype('int64'), 'loan_date': dtype('O'), 'return_date': dtype('O'), 'days_overdue': dtype('int64')}


In [4]:
# Check for the intentional data-quality issues
with engine.connect() as conn:
    # NULL cities
    null_cities = pd.read_sql(text("SELECT COUNT(*) AS null_cities FROM members WHERE city IS NULL"), conn)
    print("NULL city rows:", null_cities["null_cities"].iloc[0])

    # Negative days_overdue (data-entry error)
    bad_overdue = pd.read_sql(text("SELECT COUNT(*) AS bad_rows FROM loans WHERE days_overdue < 0"), conn)
    print("Rows with days_overdue < 0:", bad_overdue["bad_rows"].iloc[0])

    # Overdue loans (returned late)
    overdue = pd.read_sql(text("SELECT COUNT(*) AS overdue FROM loans WHERE days_overdue > 0"), conn)
    print("Overdue loans (days_overdue > 0):", overdue["overdue"].iloc[0])

NULL city rows: 30
Rows with days_overdue < 0: 10
Overdue loans (days_overdue > 0): 357


## 2 · EXTRACT

The `extract` function pulls all three tables into a Python dictionary of DataFrames.
No filtering happens here — we take the raw data exactly as it exists in the database.

In [5]:
def extract(engine):
    """
    Pull all three source tables from the database into a dictionary of DataFrames.

    Parameters
    ----------
    engine : sqlalchemy Engine

    Returns
    -------
    dict[str, pd.DataFrame]  keys: 'members', 'books', 'loans'
    """
    tables = ["members", "books", "loans"]
    data = {}
    with engine.connect() as conn:
        for table in tables:
            data[table] = pd.read_sql(text(f"SELECT * FROM {table}"), conn)
            print(f"  {table}: {len(data[table]):,} rows")
    return data


# ── Run it ───────────────────────────────────────────────────────────────────
print("Extracting...")
raw_data = extract(engine)
print("Sample — loans:")
raw_data["loans"].head()

Extracting...
  members: 100 rows
  books: 30 rows
  loans: 455 rows
Sample — loans:


,loan_id,member_id,book_id,loan_date,return_date,days_overdue
0,1,1,4,2022-08-23,2022-09-03,0
1,2,1,7,2022-12-05,2022-12-16,0
2,3,1,11,2022-10-13,2022-11-20,24
3,4,1,18,2022-05-16,2022-06-28,29
4,5,1,7,2022-09-06,2022-10-13,23


## 3 · TRANSFORM

### Join diagram
`loans` ← join on `book_id` → `books` ← join on `member_id` → `members`

### Business rules
1. **Exclude** loans where `days_overdue < 0` (data-entry errors)  
2. **Compute** `loan_duration_days = return_date - loan_date`  
3. **Compute** `overdue_rate = overdue_loans / total_loans` (proportion of loans returned late)  
4. **Aggregate** to one row per member: `total_loans`, `avg_loan_duration_days`, `overdue_loans`, `overdue_rate`, `favourite_genre`

In [6]:
def transform(data_dict):
    """
    Join tables, apply quality filters, compute loan duration and overdue rate,
    and aggregate to a member-level summary DataFrame.
    """
    members = data_dict["members"]
    books   = data_dict["books"]
    loans   = data_dict["loans"]

    # ── Step 1: filter bad loans (negative days_overdue = data-entry error) ──
    valid_loans = loans[loans["days_overdue"] >= 0].copy()
    print(f"  Valid loans after removing negative days_overdue: {len(valid_loans):,}")

    # ── Step 2: join loans → books → members ─────────────────────────────────
    df = (
        valid_loans
        .merge(books[["book_id", "genre"]],            on="book_id",   how="left")
        .merge(members[["member_id", "name", "city"]], on="member_id", how="left")
    )
    print(f"  Rows after join: {len(df):,}")

    # ── Step 3: compute loan duration ────────────────────────────────────────
    df["loan_duration_days"] = (
        pd.to_datetime(df["return_date"]) - pd.to_datetime(df["loan_date"])
    ).dt.days

    # ── Step 4: aggregate per member ─────────────────────────────────────────
    loan_counts = (
        df.groupby("member_id")
          .agg(
              total_loans            =("loan_id",            "count"),
              avg_loan_duration_days =("loan_duration_days", "mean"),
              overdue_loans          =("days_overdue",        lambda x: (x > 0).sum()),
          )
          .reset_index()
    )
    loan_counts["avg_loan_duration_days"] = loan_counts["avg_loan_duration_days"].round(1)

    # ── Step 5: compute overdue_rate (new metric) ─────────────────────────────
    loan_counts["overdue_rate"] = (
        loan_counts["overdue_loans"] / loan_counts["total_loans"]
    ).round(2)

    # ── Step 6: favourite genre per member ───────────────────────────────────
    top_genre = (
        df.groupby(["member_id", "genre"])["loan_id"]
          .count().reset_index()
          .sort_values("loan_id", ascending=False)
          .drop_duplicates("member_id")[["member_id", "genre"]]
          .rename(columns={"genre": "favourite_genre"})
    )

    # ── Step 7: join back to member info ─────────────────────────────────────
    result = (
        members[["member_id", "name", "city"]]
        .merge(loan_counts, on="member_id", how="inner")
        .merge(top_genre,   on="member_id", how="left")
    )

    print(f"  Final summary rows: {len(result):,}")
    return result


# ── Run it ───────────────────────────────────────────────────────────────────
print("Transforming...")
summary_df = transform(raw_data)
print()
summary_df.head(10)

Transforming...
  Valid loans after removing negative days_overdue: 445
  Rows after join: 445
  Final summary rows: 100



,member_id,name,city,total_loans,avg_loan_duration_days,overdue_loans,overdue_rate,favourite_genre
0,1,Member_001,Zarqa,8,24.0,4,0.50,Mystery
1,2,Member_002,Salt,2,28.5,2,1.00,History
2,3,Member_003,Aqaba,3,35.0,3,1.00,Biography
3,4,Member_004,Zarqa,8,26.4,6,0.75,Non-Fiction
4,5,Member_005,Zarqa,2,27.0,2,1.00,Mystery
5,6,Member_006,NaN,1,44.0,1,1.00,Biography
6,7,Member_007,Amman,6,21.8,3,0.50,Mystery
7,8,Member_008,Aqaba,5,28.6,4,0.80,Biography
8,9,Member_009,Amman,4,22.0,3,0.75,Non-Fiction
9,10,Member_010,Aqaba,5,26.0,4,0.80,Non-Fiction


In [7]:
# Quick distribution check — now includes overdue_rate
print(summary_df[["total_loans", "avg_loan_duration_days", "overdue_loans", "overdue_rate"]].describe().round(2))

       total_loans  avg_loan_duration_days  overdue_loans  overdue_rate
count       100.00                  100.00         100.00        100.00
mean          4.45                   26.46           3.57          0.82
std           2.25                    6.14           1.87          0.19
min           1.00                   11.50           0.00          0.00
25%           2.00                   23.00           2.00          0.67
50%           5.00                   26.20           3.50          0.83
75%           6.00                   29.85           5.00          1.00
max           8.00                   44.00           8.00          1.00


## 4 · VALIDATE

Validation is the quality gate **between transform and load**. If any check fails the pipeline stops.
Soft issues (e.g. NULL cities) emit a warning but do not halt execution.

Our hard checks:
- No duplicate `member_id` values
- No NULL `member_id` values
- `total_loans >= 1` for every row
- `avg_loan_duration_days > 0` for every row
- `overdue_rate` between 0.0 and 1.0 for every row

Soft warning:
- NULL `city` values (flagged but not a failure)

In [8]:
def validate(df):
    """
    Run data-quality checks. Raises ValueError if any hard check fails.
    Issues a warning for soft issues (e.g. NULL city).
    """
    errors = []

    dupes = df["member_id"].duplicated().sum()
    if dupes > 0:
        errors.append(f"Duplicate member_ids: {dupes}")

    null_ids = df["member_id"].isna().sum()
    if null_ids > 0:
        errors.append(f"NULL member_ids: {null_ids}")

    bad_loans = (df["total_loans"] < 1).sum()
    if bad_loans > 0:
        errors.append(f"Rows with total_loans < 1: {bad_loans}")

    bad_duration = (df["avg_loan_duration_days"] <= 0).sum()
    if bad_duration > 0:
        errors.append(f"Non-positive avg_loan_duration: {bad_duration}")

    if "overdue_rate" in df.columns:
        bad_rate = ((df["overdue_rate"] < 0) | (df["overdue_rate"] > 1)).sum()
        if bad_rate > 0:
            errors.append(f"overdue_rate out of [0, 1] range: {bad_rate}")

    if errors:
        raise ValueError("Validation failed:\n  • " + "\n  • ".join(errors))

    # Soft warning for NULL cities — not a pipeline-stopping error
    null_cities = df["city"].isna().sum()
    if null_cities > 0:
        warnings.warn(f"⚠️  {null_cities} members have NULL city — included in output but city-level analysis will be incomplete.")

    print(f"✅  Validation passed — {len(df):,} rows are clean.")


# ── Validate the real data ───────────────────────────────────────────────────
validate(summary_df)

✅  Validation passed — 100 rows are clean.


/tmp/ipykernel_353757/36597888.py:35: UserWarning: ⚠️  30 members have NULL city — included in output but city-level analysis will be incomplete.
  warnings.warn(f"⚠️  {null_cities} members have NULL city — included in output but city-level analysis will be incomplete.")


In [9]:
# ── Demonstrate what happens when validation FAILS ──────────────────────────
bad_df = summary_df.copy()
bad_df.loc[0, "total_loans"] = 0   # inject a bad value
bad_df.loc[1, "overdue_rate"] = 1.5 # inject an out-of-range overdue_rate

try:
    validate(bad_df)
except ValueError as e:
    print("❌ Caught expected error:")
    print(e)

❌ Caught expected error:
Validation failed:
  • Rows with total_loans < 1: 1
  • overdue_rate out of [0, 1] range: 1


## 5 · LOAD

The load step writes the validated DataFrame to:
1. A **PostgreSQL table** called `member_summary` (`replace` means it is recreated on every run)
2. A **CSV file** in the `outputs/` directory

After writing, the row count is read back from Postgres to confirm the write succeeded.

In [10]:
import pathlib

def load(df, engine, csv_path="outputs/member_summary.csv"):
    """
    Write the transformed summary to PostgreSQL and CSV,
    then verify the Postgres row count matches.
    """
    csv_path = pathlib.Path(csv_path)
    csv_path.parent.mkdir(parents=True, exist_ok=True)

    # Write to Postgres
    df.to_sql("member_summary", con=engine, if_exists="replace", index=False)
    print(f"  Wrote {len(df):,} rows → PostgreSQL table 'member_summary'")

    # Verify row count in Postgres
    with engine.connect() as conn:
        db_count = conn.execute(text("SELECT COUNT(*) FROM member_summary")).scalar()
    if db_count != len(df):
        raise RuntimeError(f"Load verification failed: expected {len(df)} rows, found {db_count} in DB")
    print(f"  ✅ DB row count verified: {db_count:,}")

    # Write to CSV
    df.to_csv(csv_path, index=False)
    print(f"  Wrote CSV → {csv_path}")


# ── Run it ───────────────────────────────────────────────────────────────────
load(summary_df, engine)

  Wrote 100 rows → PostgreSQL table 'member_summary'
  ✅ DB row count verified: 100
  Wrote CSV → outputs/member_summary.csv


## 6 · Verify the Outputs

Confirm both outputs exist and look correct.

In [11]:
# ── Read back from Postgres ──────────────────────────────────────────────────
with engine.connect() as conn:
    db_result = pd.read_sql(
        text("SELECT * FROM member_summary ORDER BY total_loans DESC LIMIT 10"), conn
    )

print("Top 10 members by total loans (from Postgres):")
display(db_result)

Top 10 members by total loans (from Postgres):


,member_id,name,city,total_loans,avg_loan_duration_days,overdue_loans,overdue_rate,favourite_genre
0,37,Member_037,NaN,8,26.2,7,0.88,Fiction
1,48,Member_048,Amman,8,24.4,6,0.75,Fiction
2,29,Member_029,NaN,8,29.8,6,0.75,Fiction
3,25,Member_025,NaN,8,27.5,7,0.88,Fiction
4,17,Member_017,Salt,8,30.6,7,0.88,Science
5,40,Member_040,Salt,8,25.0,5,0.62,Biography
6,4,Member_004,Zarqa,8,26.4,6,0.75,Non-Fiction
7,18,Member_018,Aqaba,8,25.9,8,1.00,Biography
8,1,Member_001,Zarqa,8,24.0,4,0.50,Mystery
9,65,Member_065,Irbid,8,29.2,5,0.62,Non-Fiction


In [12]:
# ── Read back from CSV ───────────────────────────────────────────────────────
csv_result = pd.read_csv("outputs/member_summary.csv")
print(f"CSV rows: {len(csv_result):,}")
print(f"CSV columns: {list(csv_result.columns)}")
csv_result.head()

CSV rows: 100
CSV columns: ['member_id', 'name', 'city', 'total_loans', 'avg_loan_duration_days', 'overdue_loans', 'overdue_rate', 'favourite_genre']


,member_id,name,city,total_loans,avg_loan_duration_days,overdue_loans,overdue_rate,favourite_genre
0,1,Member_001,Zarqa,8,24.0,4,0.50,Mystery
1,2,Member_002,Salt,2,28.5,2,1.00,History
2,3,Member_003,Aqaba,3,35.0,3,1.00,Biography
3,4,Member_004,Zarqa,8,26.4,6,0.75,Non-Fiction
4,5,Member_005,Zarqa,2,27.0,2,1.00,Mystery


## 7 · Run the Full Pipeline End-to-End

Now that each function works on its own, wire them together into a single `run_pipeline` call.
Each step is timed so you can spot bottlenecks as the dataset grows.

In [13]:
def run_pipeline(db_url, csv_path="outputs/member_summary.csv"):
    engine = create_engine(db_url)
    timings = {}

    print("── EXTRACT ───")
    t0 = time.perf_counter()
    raw = extract(engine)
    timings["extract"] = time.perf_counter() - t0

    print("── TRANSFORM ──")
    t0 = time.perf_counter()
    summary = transform(raw)
    timings["transform"] = time.perf_counter() - t0

    print("── VALIDATE ───")
    t0 = time.perf_counter()
    validate(summary)
    timings["validate"] = time.perf_counter() - t0

    print("── LOAD ───────")
    t0 = time.perf_counter()
    load(summary, engine, csv_path)
    timings["load"] = time.perf_counter() - t0

    total = sum(timings.values())
    print("── DONE ✅ ────")
    print()
    print("Pipeline summary")
    print(f"  Output rows  : {len(summary):,}")
    print(f"  Output columns: {list(summary.columns)}")
    print(f"  Step timings (s):")
    for step, secs in timings.items():
        print(f"    {step:<10} {secs:.3f}s")
    print(f"  Total          {total:.3f}s")

    return summary


# DB_URL was set in the Environment Setup cell above (loaded from .env)
final_df = run_pipeline(DB_URL)

── EXTRACT ───
  members: 100 rows
  books: 30 rows
  loans: 455 rows
── TRANSFORM ──
  Valid loans after removing negative days_overdue: 445
  Rows after join: 445
  Final summary rows: 100
── VALIDATE ───
✅  Validation passed — 100 rows are clean.
── LOAD ───────
  Wrote 100 rows → PostgreSQL table 'member_summary'
  ✅ DB row count verified: 100
  Wrote CSV → outputs/member_summary.csv
── DONE ✅ ────

Pipeline summary
  Output rows  : 100
  Output columns: ['member_id', 'name', 'city', 'total_loans', 'avg_loan_duration_days', 'overdue_loans', 'overdue_rate', 'favourite_genre']
  Step timings (s):
    extract    0.028s
    transform  0.041s
    validate   0.001s
    load       0.125s
  Total          0.195s


/tmp/ipykernel_353757/36597888.py:35: UserWarning: ⚠️  30 members have NULL city — included in output but city-level analysis will be incomplete.
  warnings.warn(f"⚠️  {null_cities} members have NULL city — included in output but city-level analysis will be incomplete.")


## 8 · Run the Tests

Run the unit-test suite to check your implementation.

In [14]:
!pytest test_etl_pipeline.py -v

============================= test session starts ==============================
platform linux -- Python 3.13.12, pytest-9.0.2, pluggy-1.6.0 -- /usr/bin/python3.13
cachedir: .pytest_cache
rootdir: /home/brandon/Documents/brandon/Istidama Bootcamp/Modules/Module 3 Integrated Lab/files
plugins: anyio-4.12.1
collected 20 items                                                             

test_etl_pipeline.py::TestExtract::test_returns_dict_with_three_keys PASSED [  5%]
test_etl_pipeline.py::TestExtract::test_returns_dataframes PASSED        [ 10%]
test_etl_pipeline.py::TestTransform::test_output_is_dataframe PASSED     [ 15%]
test_etl_pipeline.py::TestTransform::test_removes_negative_days_overdue PASSED [ 20%]
test_etl_pipeline.py::TestTransform::test_loan_duration_computed_correctly PASSED [ 25%]
test_etl_pipeline.py::TestTransform::test_output_columns_present PASSED  [ 30%]
test_etl_pipeline.py::TestTransform::test_one_row_per_member PASSED      [ 35%]
test_etl_pipeline.py::TestTransfo